In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [3]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

## Functions

This project required the development of six specialized scraping functions responsible for collecting match results, match events, league standings, historical standings progression, titles, and squad information. The architecture was designed to be modular and reusable, enabling data extraction from both completed and ongoing seasons through round-specific queries. Furthermore, the same functions can be easily adapted to other competitions available on Transfermarkt, as long as they share the same underlying page structure. Each function and its role within the project are presented in the following sections.

### 1) get_events()

The get_events() function extracts all match events from a specific round of a Transfermarkt competition. It identifies the team involved, the minute of the event, the player responsible, and classifies the event into predefined categories, including regular goals, penalty goals, own goals, missed penalties, and red cards. The function also generates unique identifiers for each event and accounts for different page layouts to ensure accurate data extraction. The resulting data is returned in a structured format, ready for analysis or conversion into a DataFrame.

In [8]:
def get_events(headers, league, n_season, n_round):
        # Variables Needed
        goals_list = []
        count_event = 0
        n_match = 0
        season_id = f'PL-{n_season}'
        
        # Inicializing Beautiful Soup
        url = f'https://www.transfermarkt.com.br/{league}/spieltag/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}'
        response = requests.get(url, headers=headers)
        response.status_code
        soup = BeautifulSoup(response.content, "lxml")

        # Storing all match related data in a single list
        all_matches = soup.find_all('table', {'style':'border-top: 0 !important;'})
        
        
        for match in all_matches:
            # Creating match identifier
            n_match += 1
            match_id = f'M-{n_season}-{n_round:02d}-{n_match:02d}'
            
            # Storing all events data in a single list
            event = match.find_all('tr', {'class':'no-border spieltagsansicht-aktionen'})

            # List with the entire class necessary to get the home and away team's names
            gross_h_team = match.find('td', {'class':'rechts hauptlink no-border-rechts hide-for-small spieltagsansicht-vereinsname'})
            gross_a_team = match.find('td', {'class':'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'})

            # Checking for a possible forum buttom
            home_forum_check = gross_h_team.find('a').get('href')
            away_forum_check = gross_a_team.find('a').get('href')

            # Different ways to get the title depending if it has the forum buttom
            if 'forum' in home_forum_check and 'forum' in away_forum_check:
                h_team = gross_h_team.find_all('a')[1].get('title')
                a_team = gross_a_team.find_all('a')[1].get('title')
            elif 'forum' in home_forum_check:
                h_team = gross_h_team.find_all('a')[1].get('title')
                a_team = gross_a_team.find('a').get('title')
            elif 'forum' in away_forum_check:
                h_team = gross_h_team.find('a').get('title')
                a_team = gross_a_team.find_all('a')[1].get('title')
            else:
                h_team = gross_h_team.find('a').get('title')
                a_team = gross_a_team.find('a').get('title')

            # Access one by one all match related events
            for row in event:
                # Temporary list to store events of a single match
                temp = []
                temp.append(season_id)
                temp.append(match_id)

                # Creating event identifier
                count_event += 1
                event_id = f"E-{n_season}-{n_round:02d}-{count_event:04d}"
                temp.append(event_id)

                # Transfermarkt separates home and away team events
                # Home Team Events
                try: 
                    event_type = row.find('td', {'class':'rechts no-border-rechts spieltagsansicht'}).find_all('span')[2].get('class')[1]
                    event_minute = row.find('td', {'class':'zentriert no-border-links'}).string
                    temp.append(h_team)
                    temp.append(event_minute)
                
                # Away Team Events
                except: 
                    event_type = row.find('td', {'class':'links no-border-links spieltagsansicht'}).find('span').get('class')[1]
                    event_minute = row.find('td', {'class':'zentriert no-border-rechts'}).string
                    temp.append(a_team)
                    temp.append(event_minute)

                # Event Type Information
                if event_type == 'icon-tor-formation': temp.append(0) # Normal Goal
                elif event_type == 'icon-elfmeter-formation': temp.append(1) # Penalty Goal
                elif event_type == 'icon-eigentor-formation': temp.append(2) # Own Goal
                elif event_type == 'icon-verschossener-elfmeter-formation': temp.append(-2) # Penalty Missed
                else: temp.append(-1) # Red Cards

                # Player wich made the action
                player = row.find('a').get('title')
                temp.append(player)    

                # Inserting all events related to the match into the list
                goals_list.append(temp)

        goals_list.insert(0,['season_id', 'match_id', 'event_id','goal_score_team','goal_minute','goal_type', 'goal_scorer_name'])
        return goals_list

In [5]:
events = []
head = []

for season in range(1999,2001):
    if season > 1994: season_round = 39
    else: season_round = 43

    for round in range(1,season_round+1):
        temp = get_events(headers,'premier-league',season,round)
        head=temp[0]
        events += temp[1:]
events.insert(0,head)

df_events = pd.DataFrame(events[1:], columns=events[0])
display(df_events)

,season_id,match_id,event_id,goal_score_team,goal_minute,goal_type,goal_scorer_name
0,PL-1999,M-1999-01,E-1999-01-0001,Leicester City FC,57',0,Tony Cottee
1,PL-1999,M-1999-01,E-1999-01-0002,FC Arsenal,65',0,Dennis Bergkamp
2,PL-1999,M-1999-01,E-1999-01-0003,FC Arsenal,90',2,Frank Sinclair
3,PL-1999,M-1999-02,E-1999-01-0004,Chelsea FC,20',0,Gustavo Poyet
4,PL-1999,M-1999-02,E-1999-01-0005,Chelsea FC,32',0,Gianfranco Zola
...,...,...,...,...,...,...,...
2196,PL-2000,M-2000-09,E-2000-38-0032,FC Southampton,89',0,Matt Le Tissier
2197,PL-2000,M-2000-10,E-2000-38-0033,Tottenham Hotspur,17',0,Willem Korsten
2198,PL-2000,M-2000-10,E-2000-38-0034,Manchester United FC,22',0,Paul Scholes
2199,PL-2000,M-2000-10,E-2000-38-0035,Tottenham Hotspur,67',0,Willem Korsten


### 2) get_match()

The get_match() function scrapes match-level information from a specific round of a Transfermarkt competition. It collects the participating teams, final score, match date, referee, attendance, and generates unique identifiers for the season, round, and match. The function also handles different page layouts caused by the presence of forum links, ensuring that team names are extracted correctly. Finally, the collected data is organized into a structured list, making it ready for further processing or conversion into a DataFrame.

In [6]:
def get_match(headers, league, n_season, n_round):
    all_rounds = []
    n_match = 0

    url = f'https://www.transfermarkt.com.br/{league}/spieltag/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "lxml")
    
    all_information = soup.find_all('table', {'style':'border-top: 0 !important;'})

    # Gathering Useful Information
    for row in all_information:
        temp = []
        n_match += 1

        season_key = f'PL-{n_season}'
        match_key = f'M-{n_season}-{n_round:02d}-{n_match:03d}'

        if n_round < 10: round_key = f'R-{n_season}-0' + str(n_round)
        else: round_key = f'R-{n_season}-' + str(n_round)

        temp.append(season_key)
        temp.append(round_key)
        temp.append(match_key)

        # List with the entire class necesaire to get the home and away team's names
        gross_home_team = row.find('td', {'class':'rechts hauptlink no-border-rechts hide-for-small spieltagsansicht-vereinsname'})
        gross_away_team = row.find('td', {'class':'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'})

        # Checking for a possible forum buttom
        home_forum_check = gross_home_team.find('a').get('href')
        away_forum_check = gross_away_team.find('a').get('href')

        # Different ways to get the title depending if it has the forum buttom
        if 'forum' in home_forum_check and 'forum' in away_forum_check:
            home_team = gross_home_team.find_all('a')[1].get('title')
            away_team = gross_away_team.find_all('a')[1].get('title')
        elif 'forum' in home_forum_check:
            home_team = gross_home_team.find_all('a')[1].get('title')
            away_team = gross_away_team.find('a').get('title')
        elif 'forum' in away_forum_check:
            home_team = gross_home_team.find('a').get('title')
            away_team = gross_away_team.find_all('a')[1].get('title')
        else:
            home_team = gross_home_team.find('a').get('title')
            away_team = gross_away_team.find('a').get('title')

        # Getting the final score
        final_score = row.find('span', {'class':'matchresult finished'}).string

        # Appending data from a single match together
        temp.append(home_team)
        temp.append(final_score)
        temp.append(away_team)

        # Storing adicional info separately, easier to extract right information
        adicional_info = row.find_all('td', {'class':'zentriert no-border'})

        for i, item in enumerate(adicional_info):
            if i == 2:
                text = item.get_text(" ", strip=True)
                try: 
                    attendance = text.split()[0]
                    temp.append(attendance)
                except: temp.append(text)
            else:
                day_ref = item.find('a').string
                temp.append(day_ref.strip())

        all_rounds.append(temp)

    all_rounds.insert(0,['season_id','round_id', 'match_id', 'home_team', 'final_score', 'away_team', 'date', 'referee', 'attendance'])
    return all_rounds

In [7]:
premier_matches = []
head = []

for season in range(2000,2002):
    if season > 1994: season_round = 38
    else: season_round = 42

    for round in range(1,season_round+1):
        temp = get_match(headers,'premier-league',season,round)
        premier_matches += temp[1:]
        head = temp[0]
premier_matches.insert(0,head)

df_premier_matches = pd.DataFrame(premier_matches[1:], columns=premier_matches[0])
display(df_premier_matches)

,season_id,round_id,match_id,home_team,final_score,away_team,date,referee,attendance
0,PL-2000,R-2000-01,M-2000-01-001,Charlton Athletic,4:0,Manchester City FC,19/08/2000,Rob Harris,20.043
1,PL-2000,R-2000-01,M-2000-01-002,Chelsea FC,4:2,West Ham United,19/08/2000,Graham Barber,34.914
2,PL-2000,R-2000-01,M-2000-01-003,Coventry City,1:3,FC Middlesbrough,19/08/2000,Barry Knight,20.624
3,PL-2000,R-2000-01,M-2000-01-004,Derby County,2:2,FC Southampton,19/08/2000,Andy D'Urso,27.223
4,PL-2000,R-2000-01,M-2000-01-005,Leeds United FC,2:0,FC Everton,19/08/2000,Dermot Gallagher,40.010
...,...,...,...,...,...,...,...,...,...
455,PL-2001,R-2001-38,M-2001-38-006,Leeds United FC,1:0,FC Middlesbrough,11/05/2002,Uriah Rennie,40.218
456,PL-2001,R-2001-38,M-2001-38-007,Leicester City FC,2:1,Tottenham Hotspur,11/05/2002,David Elleray,21.716
457,PL-2001,R-2001-38,M-2001-38-008,AFC Sunderland,1:1,Derby County,11/05/2002,Alan Wiley,47.989
458,PL-2001,R-2001-38,M-2001-38-009,Manchester United FC,0:0,Charlton Athletic,11/05/2002,Graham Poll,67.571


### 3) get_placements()

The get_placements() function retrieves the league standings for a specific round of a Transfermarkt competition. It extracts each team's position, matches played, wins, draws, losses, goals scored, goal difference, and total points. Additionally, the function generates unique identifiers for the season and round, organizing the collected information into a structured dataset that can be easily analyzed or converted into a DataFrame.

In [16]:
def get_placements(headers, league, n_season, n_round):
    round_classification = []
    
    url = f'https://www.transfermarkt.com.br/{league}/spieltagtabelle/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}'
    response = requests.get(url,headers=headers)
    soup = BeautifulSoup(response.content,'lxml')

    info = soup.find_all('tbody')
    table_info = info[2].find_all('tr')

    for i,row in enumerate(table_info):
        temp = []

        season_key = f'PL-{n_season}'
        round_key = f'R-{n_season}-{n_round:02d}'
        
        temp.append(season_key)
        temp.append(round_key)

        placement = i+1
        team = row.find('a').get('title')

        temp.append(placement)
        temp.append(team)

        adicional_info = row.find_all('td', {'class':'zentriert'})

        for i, item in enumerate(adicional_info):
            if i == 0: continue
            temp.append(item.string)

        round_classification.append(temp)

    round_classification.insert(0,['season_id','round_id','placement','team_name','matches','wins','draws','losses','goals','goal_dif','points'])
    return round_classification

In [17]:
premier_placements = []
head = []

for season in range(2000,2002):
    if season > 1994: season_round = 38
    else: season_round = 42

    for round in range(1,season_round+1):
        temp = get_placements(headers,'premier-league',season,round)
        premier_placements += temp[1:]
        head = temp[0]
premier_placements.insert(0,head)

df_premier_placements = pd.DataFrame(premier_placements[1:], columns=premier_placements[0])
display(df_premier_placements)

,season_id,round_id,placement,team_name,matches,wins,draws,losses,goals,goal_dif,points
0,PL-2000,R-2000-01,1,Charlton Athletic,1,1,0,0,4:0,4,3
1,PL-2000,R-2000-01,2,Chelsea FC,1,1,0,0,4:2,2,3
2,PL-2000,R-2000-01,3,FC Middlesbrough,1,1,0,0,3:1,2,3
3,PL-2000,R-2000-01,4,Tottenham Hotspur,1,1,0,0,3:1,2,3
4,PL-2000,R-2000-01,5,Leeds United FC,1,1,0,0,2:0,2,3
...,...,...,...,...,...,...,...,...,...,...,...
1515,PL-2001,R-2001-38,16,Bolton Wanderers,38,9,13,16,44:62,-18,40
1516,PL-2001,R-2001-38,17,AFC Sunderland,38,10,10,18,29:51,-22,40
1517,PL-2001,R-2001-38,18,Ipswich Town FC,38,9,9,20,41:64,-23,36
1518,PL-2001,R-2001-38,19,Derby County,38,8,6,24,33:63,-30,30


### 4) get_squad()

The get_squad() function extracts squad-related information for every team participating in a Transfermarkt competition during a given season. It retrieves each team's market value, squad size, average player age, and number of foreign players. The function also handles slight variations in the page structure when extracting market values, ensuring consistent results. All collected information is organized into a structured dataset that can be easily analyzed or converted into a DataFrame.

In [5]:
def get_squad(headers, league, n_season):
    value = []

    url = f'https://www.transfermarkt.com.br/{league}/startseite/wettbewerb/GB1/plus/?saison_id={n_season}'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "lxml")

    tables = soup.find_all('table', {'class':'items'})
    main_table = tables[0]

    even_info = main_table.find_all('tr', {'class':'even'})
    odd_info = main_table.find_all('tr', {'class':'odd'})

    info = odd_info + even_info

    season_key = f'PL-{n_season}'

    for row in info:
        temp = []

        temp.append(season_key)

        team_name = row.find('a').get('title')
        
        if row.find_all('a')[2].get('href') == '#': team_value = row.find_all('a')[-1].string
        else: team_value = row.find_all('a')[3].string

        temp.append(team_name)
        temp.append(team_value)

        squad_info = row.find_all('td', {'class':'zentriert'})
        for i, item in enumerate(squad_info):
            if i != 0: temp.append(item.string)

        value.append(temp)

    value.insert(0, ['season_id', 'team_name','team_value','team_squad','team_avg_age','team_foreigners'])
    return value

In [ ]:
premier_squad = []
head = []

# transfermarkt only contains values for team_value from 2004
for season in range(2004,2026):
    temp = get_squad(headers,'premier-league',season)
    premier_squad += temp[1:]
    head = temp[0]
premier_squad.insert(0,head)

df_premier_squad = pd.DataFrame(premier_squad[1:], columns=premier_squad[0])
display(df_premier_squad)

,season_id,team_name,team_value,team_squad,team_avg_age,team_foreigners
0,PL-2004,Chelsea FC,€ 331.48 mi.,31,24.9,24
1,PL-2004,FC Arsenal,€ 247.00 mi.,37,23.9,29
2,PL-2004,Tottenham Hotspur,€ 129.45 mi.,38,25.3,22
3,PL-2004,Birmingham City,€ 106.93 mi.,38,26.8,21
4,PL-2004,Manchester City FC,€ 94.30 mi.,34,26.7,21
...,...,...,...,...,...,...
435,PL-2025,Aston Villa FC,€ 531.50 mi.,26,28.2,17
436,PL-2025,FC Everton,€ 443.15 mi.,26,26.8,13
437,PL-2025,West Ham United,€ 362.35 mi.,23,27.1,17
438,PL-2025,FC Fulham,€ 356.20 mi.,25,28.1,21


### 5) get_title()

The get_title() function retrieves the historical champions of a Transfermarkt competition. For each title-winning season, it extracts the champion club and its manager, while also converting the season label into a standardized season identifier. In the case of the Premier League, the function considers only seasons from 1992 onward, when the competition adopted its current format. The collected data is returned as a structured dataset, ready for analysis or conversion into a DataFrame.

In [4]:
def get_title(headers, league):
    titles = []

    url = f'https://www.transfermarkt.com.br/{league}/erfolge/wettbewerb/GB1'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "lxml")

    all_info = soup.find_all('tbody')
    info = all_info[0].find_all('tr')

    for row in info:
        temp = []

        season = row.find('td', {'class':'zentriert'}).string
        if season == '91/92': break # First season of the current format of the Premier League, maybe add a parameter to stop
        
        x = int(season.strip('/')[0] + season.strip('/')[1])
        if x > 90: n_season = x+1900
        else: n_season = x+2000
        season_key = f'PL-{n_season}'
        
        team_manager = row.find_all('a')
        temp.append(season_key)
        temp.append(season)

        for i, item in enumerate(team_manager):
            if i == 0: continue
            temp.append(item.string)
        
        titles.append(temp)

    titles.insert(0,['season_id', 'season_name','team_name', 'manager_name'])
    return titles

In [5]:
premier_titles = get_title(headers,'premier-league')

df_premier_titles = pd.DataFrame(premier_titles[1:], columns=premier_titles[0])
display(df_premier_titles)

,season_id,season_name,team_name,manager_name
0,PL-2025,25/26,FC Arsenal,Mikel Arteta
1,PL-2024,24/25,FC Liverpool,Arne Slot
2,PL-2023,23/24,Manchester City FC,Pep Guardiola
3,PL-2022,22/23,Manchester City FC,Pep Guardiola
4,PL-2021,21/22,Manchester City FC,Pep Guardiola
5,PL-2020,20/21,Manchester City FC,Pep Guardiola
6,PL-2019,19/20,FC Liverpool,Jürgen Klopp
7,PL-2018,18/19,Manchester City FC,Pep Guardiola
8,PL-2017,17/18,Manchester City FC,Pep Guardiola
9,PL-2016,16/17,Chelsea FC,Antonio Conte
